# ISP Pipeline Inference

Full forward pipeline over a RAW image:

```
RAW (16-bit Bayer PNG + JSON, 6144×8192)
 -> InputRawBlock
 -> BlackLevelCorrectionBlock
 -> NormalizationBlock
 -> JDDBlock               (trained JDD / NAFNet: joint demosaic + denoise)
 -> AutoWhiteBalanceBlock  (as_shot_neutral)
 -> XYZTransformBlock
 -> SRGBTransformBlock
 -> UnrealToneMappingBlock
 -> GammaCorrectionBlock
 -> SpatialTransformBlock
 -> HDRNetBlock
```

**How to use:**

1. In the "Configuration" cell, set `INPUT_DIR` (folder with raw frames `{i}.png` + `{i}.json`),
   `OUTPUT_DIR` (where to save the results) and, if needed, `GT_DIR`
   (folder with ground truth `{i}.JPG` — used only for PSNR/SSIM metrics).
2. Run all cells top to bottom.
3. Results: PNG frames `{i}.png` + `metrics.csv` in `OUTPUT_DIR`,
   a PSNR/SSIM table and summary statistics in the cell outputs.

Input data format:
16-bit grayscale RAW PNG + JSON metadata. `data/` contains
1 demo frame (900) with ground truth.


In [ ]:
import os
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

PACKAGE_DIR = os.path.abspath(".")
if PACKAGE_DIR not in sys.path:
    sys.path.insert(0, PACKAGE_DIR)

from src.blocks.base_pipeline_blocks import (
    AutoWhiteBalanceBlock,
    BlackLevelCorrectionBlock,
    GammaCorrectionBlock,
    JDDBlock,
    NormalizationBlock,
    SpatialTransformBlock,
    SRGBTransformBlock,
    XYZTransformBlock,
)
from src.blocks.hdrnet_block import HDRNetBlock
from src.blocks.input_raw_block import InputRawBlock
from src.blocks.tone_mapping_blocks import UnrealToneMappingBlock
from src.metrics import psnr_metric, ssim_metric
from src.nafnet.inference import load_checkpoint, nafnet_jdd_default
from src.nafnet.rggb_model_wrapper import RGGBModelWrapper
from src.hdrnet.model import HDRPointwiseNN

DEVICE = torch.device("cpu")

np.seterr(all="ignore")

print("torch :", torch.__version__)
print("device:", DEVICE)
print("pkg   :", PACKAGE_DIR)


## Run Configuration

In [ ]:
# ---- Run configuration ----

# INPUT_DIR: folder with raw frames: {i}.png (16-bit Bayer) + {i}.json (metadata).
INPUT_DIR = os.path.join(PACKAGE_DIR, "data", "raw")

# GT_DIR: folder with ground truth ({i}.JPG). Needed only for PSNR/SSIM computation;
GT_DIR = os.path.join(PACKAGE_DIR, "data", "ground_truth")

# OUTPUT_DIR: where to save the final PNGs and metrics.csv.
OUTPUT_DIR = os.path.join(PACKAGE_DIR, "output")

# FRAME_RANGE: None -> process all frames found in INPUT_DIR;
#              (start, end) -> only indices in the range [start, end] inclusive.
FRAME_RANGE = None

SAVE_IMAGES = True       # save final PNGs
COMPUTE_METRICS = True   # compute PSNR/SSIM against GT (requires GT_DIR)
SKIP_EXISTING = False    # skip already saved frames (requires SAVE_IMAGES)
SAVE_METRICS_CSV = True  # write metrics.csv to OUTPUT_DIR

RAW_BLOCK = InputRawBlock(INPUT_DIR)

if FRAME_RANGE is None:
    FRAMES = RAW_BLOCK.available_indices()
else:
    start, end = FRAME_RANGE
    FRAMES = [i for i in RAW_BLOCK.available_indices() if start <= i <= end]
assert FRAMES, f"No frames to process found in {INPUT_DIR}"

if SAVE_IMAGES or SAVE_METRICS_CSV:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print("INPUT_DIR  :", RAW_BLOCK.inputs_path)
print("GT_DIR     :", GT_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("FRAME_RANGE:", FRAME_RANGE if FRAME_RANGE is not None else "(all found)")
if len(FRAMES) <= 20:
    print("frames    :", FRAMES)
else:
    print(f"frames    : {len(FRAMES)} ({FRAMES[0]}..{FRAMES[-1]})")


## JDD Weights (NAFNet)

In [ ]:
# ---- JDD (NAFNet) weights: joint demosaic + denoise ----
JDD_CKPT = os.path.join(PACKAGE_DIR, "checkpoints", "jdd", "nafnet_jdd_epoch_175.pt")

# Architecture is fixed — as in the training of this checkpoint.
JDD_MODEL = RGGBModelWrapper(
    nafnet_jdd_default(img_channel=16, output_channels=16), 4, 12, 16
).to(DEVICE)
load_checkpoint(JDD_MODEL, JDD_CKPT, DEVICE)
JDD_MODEL.eval()

print("__ JDD WEIGHTS IN USE __")
print("  path   :", JDD_CKPT)
print("  exists :", os.path.isfile(JDD_CKPT))
print("  device :", next(JDD_MODEL.parameters()).device)


## HDRNet Weights

In [ ]:
# ---- HDRNet weights ----
HDRNET_CKPT = os.path.join(PACKAGE_DIR, "checkpoints", "hdrnet", "epoch30.pth")


def load_ckpt_params(path):
    ckpt = torch.load(path, map_location="cpu")
    cfg = ckpt.get("config") or {}
    params = dict(cfg.get("model") or ckpt.get("params") or {})
    dataset = dict(cfg.get("dataset") or {})
    return ckpt, params, dataset


ckpt, HDRNET_PARAMS, HDRNET_DATASET = load_ckpt_params(HDRNET_CKPT)
assert HDRNET_PARAMS, f"No model parameters found in checkpoint {HDRNET_CKPT}"
LOWRES_SIZE = int(HDRNET_PARAMS.get("net_input_size", 256))
FULLRES_SIZE = int(HDRNET_PARAMS.get("net_output_size", 2000))

HDRNET_MODEL = HDRPointwiseNN(params=HDRNET_PARAMS)
HDRNET_MODEL.load_state_dict(ckpt["state_dict"])
HDRNET_MODEL.to(DEVICE).eval()

print("__ HDRNet WEIGHTS IN USE __")
print("  path            :", HDRNET_CKPT)
print("  exists          :", os.path.isfile(HDRNET_CKPT))
print("  net_input_size  :", LOWRES_SIZE)
print("  net_output_size :", FULLRES_SIZE)
print("  luma_bins       :", HDRNET_PARAMS.get("luma_bins"))
print("  channel_multiplier:", HDRNET_PARAMS.get("channel_multiplier"))
print("  spatial_bin     :", HDRNET_PARAMS.get("spatial_bin"))
print("  guide_complexity:", HDRNET_PARAMS.get("guide_complexity"))
print("  device          :", next(HDRNET_MODEL.parameters()).device)


## Color and Geometry Matrices

In [ ]:
# ---- Color transformation and geometry parameters ----
COLOR_MATRIX = np.array(
    [
        [1.06835938, -0.29882812, -0.14257812],
        [-0.43164062, 1.35546875, 0.05078125],
        [-0.1015625, 0.24414062, 0.5859375],
    ]
)

XYZ2SRGB = np.array(
    [
        [3.2404542, -1.5371385, -0.4985314],
        [-0.9692660, 1.8760108, 0.0415560],
        [0.0556434, -0.2040259, 1.0572252],
    ]
)

TRANSFORMATION_MATRIX = np.array(
    [
        [4.6784997e00, 5.3825771e-04, -2.6050076e02],
        [9.1054139e-04, 4.6764960e00, -3.4387198e02],
        [5.3933155e-07, 2.7009705e-07, 1.0000000e00],
    ]
) @ np.diag([1 / 2, 1 / 2, 1.0])

TONE_MAPPING_SLOPE = 1.15
TONE_MAPPING_OFFSET = 0.155

print("COLOR_MATRIX        :", COLOR_MATRIX.shape)
print("XYZ2SRGB            :", XYZ2SRGB.shape)
print("TRANSFORMATION_MATRIX:", TRANSFORMATION_MATRIX.shape)
print("TONE_MAPPING         : slope =", TONE_MAPPING_SLOPE, ", offset =", TONE_MAPPING_OFFSET)


## Pipeline Construction

In [ ]:
# ---- Pipeline construction ----


def build_pipeline(
    jdd_model=JDD_MODEL,
    hdrnet_model=HDRNET_MODEL,
    lowres_size=LOWRES_SIZE,
    device=DEVICE,
):
    block = InputRawBlock(INPUT_DIR)
    block = BlackLevelCorrectionBlock(block)
    block = NormalizationBlock(block)
    block = JDDBlock(block, jdd_model, device)
    block = AutoWhiteBalanceBlock(block, illumination_estimation_option=None)
    block = XYZTransformBlock(block, color_matrix=COLOR_MATRIX)
    block = SRGBTransformBlock(block, xyz_to_srgb_matrix=XYZ2SRGB)
    block = UnrealToneMappingBlock(
        block, slope=TONE_MAPPING_SLOPE, offset=TONE_MAPPING_OFFSET
    )
    block = GammaCorrectionBlock(block)
    block = SpatialTransformBlock(block, transform_matrix=TRANSFORMATION_MATRIX)
    block = HDRNetBlock(
        block, model=hdrnet_model, device=device, lowres_size=lowres_size
    )
    return block


def to_uint8(image):
    return (np.clip(image, 0.0, 1.0) * 255.0).astype(np.uint8)


PIPELINE = build_pipeline()

print("pipeline length:", len(PIPELINE))
print("frames to process:", FRAMES)


## Single Frame: GT vs Prediction

In [ ]:
# ---- Single frame: GT vs prediction ----
# SAMPLE_INDEX must be one of FRAMES.
SAMPLE_INDEX = FRAMES[0]

sample_start = time.perf_counter()
pred = PIPELINE[SAMPLE_INDEX]["image"]
elapsed = time.perf_counter() - sample_start

pred_u8 = to_uint8(pred)

psnr_v, ssim_v = float("nan"), float("nan")
gt_u8 = None
if COMPUTE_METRICS and GT_DIR:
    gt_u8 = cv2.imread(
        os.path.join(GT_DIR, f"{SAMPLE_INDEX}.JPG"), cv2.IMREAD_COLOR_RGB
    )
    if gt_u8 is None:
        print(f"GT not found for frame {SAMPLE_INDEX} — metrics skipped")
    elif gt_u8.shape != pred_u8.shape:
        print(
            f"Shape mismatch: pred={pred_u8.shape}, gt={gt_u8.shape} — metrics skipped"
        )
        gt_u8 = None
    else:
        psnr_v = psnr_metric(pred_u8, gt_u8)
        ssim_v = ssim_metric(pred_u8, gt_u8)

print(f"sample {SAMPLE_INDEX}: pred {pred_u8.shape}, {elapsed:.1f}s")
if gt_u8 is not None:
    print(f"  PSNR {psnr_v:.2f} dB   SSIM {ssim_v:.4f}")
    panels = [
        ("GT", gt_u8),
        (f"Predict (PSNR {psnr_v:.2f} dB, SSIM {ssim_v:.4f})", pred_u8),
    ]
else:
    panels = [(f"Predict (HDRNet last, {elapsed:.1f}s)", pred_u8)]

fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 7))
for ax, (title, img) in zip(np.atleast_1d(axes), panels):
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Batch Run: PNG Saving and PSNR/SSIM

In [ ]:
# ---- Batch run: save PNGs + PSNR/SSIM against GT ----
psnrs, ssims = [], []
rows, processed, skipped = [], [], []

header = f"{'frame':>6} {'PSNR':>8} {'SSIM':>8} {'seconds':>8}"
print(header)
print("-" * len(header))

total_start = time.perf_counter()
for i in FRAMES:
    out_path = os.path.join(OUTPUT_DIR, f"{i}.png")
    if SKIP_EXISTING and SAVE_IMAGES and os.path.exists(out_path):
        skipped.append(i)
        continue

    frame_start = time.perf_counter()
    pred = PIPELINE[i]["image"]
    pred_u8 = to_uint8(pred)
    frame_time = time.perf_counter() - frame_start

    if SAVE_IMAGES:
        cv2.imwrite(out_path, cv2.cvtColor(pred_u8, cv2.COLOR_RGB2BGR))

    psnr_v, ssim_v = float("nan"), float("nan")
    if COMPUTE_METRICS and GT_DIR:
        gt_u8 = cv2.imread(os.path.join(GT_DIR, f"{i}.JPG"), cv2.IMREAD_COLOR_RGB)
        if gt_u8 is None:
            print(f"{i:>6}  GT not found, metrics skipped")
        elif gt_u8.shape != pred_u8.shape:
            print(
                f"{i:>6}  shape mismatch pred={pred_u8.shape} gt={gt_u8.shape}, metrics skipped"
            )
        else:
            psnr_v = psnr_metric(pred_u8, gt_u8)
            ssim_v = ssim_metric(pred_u8, gt_u8)
            psnrs.append(psnr_v)
            ssims.append(ssim_v)

    processed.append(i)
    rows.append({"frame": i, "psnr": psnr_v, "ssim": ssim_v, "seconds": frame_time})
    print(f"{i:>6} {psnr_v:>8.2f} {ssim_v:>8.4f} {frame_time:>8.2f}")

elapsed = time.perf_counter() - total_start
print(f"\nprocessed {len(processed)} frames in {elapsed:.1f}s, skipped {len(skipped)}")

if psnrs:
    arr_p, arr_s = np.array(psnrs), np.array(ssims)
    print(
        f"PSNR: mean={arr_p.mean():.2f} dB  std={arr_p.std():.2f}  "
        f"min={arr_p.min():.2f}  max={arr_p.max():.2f}"
    )
    print(
        f"SSIM: mean={arr_s.mean():.4f}    std={arr_s.std():.4f}  "
        f"min={arr_s.min():.4f}  max={arr_s.max():.4f}"
    )

if SAVE_METRICS_CSV and rows:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, "metrics.csv")
    with open(csv_path, "w") as f:
        f.write("frame,psnr,ssim,seconds\n")
        for row in rows:
            f.write(
                f"{row['frame']},{row['psnr']:.4f},{row['ssim']:.6f},"
                f"{row['seconds']:.2f}\n"
            )
    print("metrics saved to:", csv_path)

if SAVE_IMAGES:
    print("results saved to:", OUTPUT_DIR)
